# Cross-Model Embedding Comparison

**AI-2 Session 2.1 — Standalone Reference**

---

This notebook compares Voyage AI embedding models on the Northbrook Partners corpus. It is a self-contained copy of the cross-model comparison section from `session_2_1.ipynb`. You can use it independently to revisit the comparison without opening the full session notebook.

**What this notebook does:**
1. Loads the Northbrook Partners document corpus and chunks it
2. Embeds every chunk with three Voyage AI models (`voyage-3-lite`, `voyage-3`, `voyage-3-large`)
3. Runs a retrieval test using queries with known relevant documents
4. Compares models on dimensions, embedding time, retrieval accuracy, and similarity scores
5. Visualizes results and estimates cost per 1,000 documents

**Prerequisites:**
- `VOYAGE_API_KEY` set in your `.env` file
- All project dependencies installed (`uv sync`)

---

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv()

import os
import time
import pathlib
import numpy as np
import matplotlib.pyplot as plt

from src.s2_embeddings.embed import get_embedding, embed_texts, cosine_similarity
from src.s3_ingestion.chunker import chunk_document

# Verify Voyage API key is loaded
key = os.getenv("VOYAGE_API_KEY")
if key:
    print(f"VOYAGE_API_KEY loaded (starts with {key[:8]}...)")
else:
    print("ERROR: No VOYAGE_API_KEY found. Check your .env file.")

In [ ]:
# Models to compare
MODELS = {
    "voyage-3-lite": {
        "dimensions": 1024,
        "tier": "lite",
        "cost_per_million_tokens": 0.02,
    },
    "voyage-3": {
        "dimensions": 1024,
        "tier": "standard",
        "cost_per_million_tokens": 0.06,
    },
    "voyage-3-large": {
        "dimensions": 2048,
        "tier": "large",
        "cost_per_million_tokens": 0.18,
    },
}

MODEL_NAMES = list(MODELS.keys())
print(f"Models to compare: {MODEL_NAMES}")

---

## 2. Define Test Queries

Each query has one or more expected relevant documents. We use these to measure whether each model's embeddings can retrieve the correct source.

In [ ]:
test_queries = [
    # --- EASY: Direct keyword overlap, all models should hit ---
    {
        "query": "What is the vacation policy?",
        "relevant_sources": ["vacation_policy_2025.md"],
        "difficulty": "easy",
    },
    {
        "query": "When is the office moving?",
        "relevant_sources": ["memo_office_relocation.md"],
        "difficulty": "easy",
    },
    {
        "query": "What are the MFA requirements?",
        "relevant_sources": ["memo_security_update.md"],
        "difficulty": "easy",
    },
    {
        "query": "What was Q3 revenue?",
        "relevant_sources": ["financial_report_q3_2024.md"],
        "difficulty": "easy",
    },

    # --- MEDIUM: Paraphrased or indirect — smaller models may struggle ---
    {
        "query": "How much can I spend on dinner when traveling for work?",
        "relevant_sources": ["expense_policy.md"],
        "difficulty": "medium",
    },
    {
        "query": "What happens to my unused PTO days at the end of the year?",
        "relevant_sources": ["vacation_policy_2025.md", "memo_policy_correction.md"],
        "difficulty": "medium",
    },
    {
        "query": "Who is leading the AI initiative and what is the budget?",
        "relevant_sources": ["board_meeting_q4_2024.md", "memo_ceo_annual_kickoff.md"],
        "difficulty": "medium",
    },
    {
        "query": "What specialized engineers is the company trying to hire?",
        "relevant_sources": ["engineering_standup_2025_01.md"],
        "difficulty": "medium",
    },

    # --- HARD: Vocabulary mismatch, cross-document, nuanced ---
    {
        "query": "Why is the company investing in employee well-being this year?",
        "relevant_sources": ["vacation_policy_2025.md", "hr_committee_2025_01.md", "memo_ceo_annual_kickoff.md"],
        "difficulty": "hard",
    },
    {
        "query": "What is the target leverage ratio and who proposed it?",
        "relevant_sources": ["board_meeting_q4_2024.md"],
        "difficulty": "hard",
    },
    {
        "query": "How has client concentration risk changed over the past year?",
        "relevant_sources": ["financial_report_q4_2024.md", "board_meeting_q4_2024.md", "board_meeting_q3_2024.md"],
        "difficulty": "hard",
    },
    {
        "query": "What caused the decline in profit margins last quarter?",
        "relevant_sources": ["financial_report_q3_2024.md"],
        "difficulty": "hard",
    },
    {
        "query": "Which infrastructure modernization project is behind schedule?",
        "relevant_sources": ["engineering_standup_2025_01.md"],
        "difficulty": "hard",
    },
    {
        "query": "What is the lowest-scoring category in employee satisfaction?",
        "relevant_sources": ["hr_committee_2025_01.md"],
        "difficulty": "hard",
    },
]

print(f"Defined {len(test_queries)} test queries:")
print(f"  Easy:   {sum(1 for q in test_queries if q['difficulty'] == 'easy')}")
print(f"  Medium: {sum(1 for q in test_queries if q['difficulty'] == 'medium')}")
print(f"  Hard:   {sum(1 for q in test_queries if q['difficulty'] == 'hard')}")
print()
for tq in test_queries:
    print(f"  [{tq['difficulty']:>6}] {tq['query']!r}")
    print(f"          --> {tq['relevant_sources']}")

---

## 3. Load Corpus

Load all Northbrook Partners documents from the data directory, then chunk them using the paragraph-aware chunker. Each chunk carries metadata including its source filename.

In [ ]:
DATA_DIR = pathlib.Path("../data/northbrook")

# Load all markdown files from the corpus directory
documents = []
for filepath in sorted(DATA_DIR.glob("*.md")):
    text = filepath.read_text(encoding="utf-8")
    documents.append({
        "filename": filepath.name,
        "text": text,
    })

print(f"Loaded {len(documents)} documents:")
for doc in documents:
    print(f"  - {doc['filename']} ({len(doc['text']):,} chars)")

In [ ]:
# Chunk every document using the paragraph-aware chunker
all_chunks = []
for doc in documents:
    chunks = chunk_document(
        text=doc["text"],
        source=doc["filename"],
        chunk_size=500,
        overlap=100,
        doc_type="northbrook",
    )
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")
print()
# Show a sample chunk
sample = all_chunks[0]
print("Sample chunk metadata:", sample.metadata)
print(f"Sample chunk text (first 200 chars): {sample.text[:200]}...")

---

## 4. Embed with Each Model

For each of the three Voyage AI models, embed all corpus chunks and record how long it takes. The embeddings are stored in a dictionary keyed by model name.

In [ ]:
chunk_texts = [c.text for c in all_chunks]

# Cache directory for saved embeddings
CACHE_DIR = pathlib.Path("cache/embeddings")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Store embeddings and timing for each model
model_embeddings = {}  # model_name -> list of embedding vectors
model_times = {}       # model_name -> seconds to embed corpus

for model_name in MODEL_NAMES:
    cache_file = CACHE_DIR / f"{model_name.replace('-', '_')}_{len(chunk_texts)}_chunks.npz"
    
    if cache_file.exists():
        # Load from cache
        data = np.load(cache_file)
        embeddings = data["embeddings"].tolist()
        model_embeddings[model_name] = embeddings
        model_times[model_name] = float(data["elapsed"])
        print(f"{model_name}: loaded from cache  (dim={len(embeddings[0])}, original time: {model_times[model_name]:.2f}s)")
    else:
        # Embed fresh and save to cache
        print(f"Embedding {len(chunk_texts)} chunks with {model_name}...", end=" ", flush=True)
        start = time.time()
        embeddings = embed_texts(chunk_texts, model=model_name)
        elapsed = time.time() - start
        
        model_embeddings[model_name] = embeddings
        model_times[model_name] = elapsed
        
        # Save to cache
        np.savez(cache_file, embeddings=np.array(embeddings), elapsed=np.array(elapsed))
        print(f"done in {elapsed:.2f}s  (dim={len(embeddings[0])}) — cached to {cache_file.name}")

print()
print("All models ready.")
print(f"Cache location: {CACHE_DIR.resolve()}")
print("To force re-embedding, delete the cache directory and re-run this cell.")

---

## 5. Retrieval Test

For each model and each test query, we embed the query, find the most similar chunks, and check whether the correct source document shows up.

### Metrics We'll Use

**Top-1 Accuracy** — "Did the model get it right on the first try?"
- We look at the single highest-scoring chunk. If it comes from the correct document, that's a hit.
- This is the strictest test. A model that nails rank 1 is more useful in production because your RAG pipeline typically feeds the top result to the LLM first.

**Top-5 Accuracy** — "Did the correct document appear anywhere in the top 5?"
- More lenient. On a small corpus like ours (~100 chunks), most models will score 100% here.
- Useful as a baseline, but doesn't tell you much about ranking *quality*.

**Mean Reciprocal Rank (MRR)** — "On average, how far down do you have to look to find the right answer?"
- If the right answer is at rank 1, the reciprocal rank is 1/1 = **1.0**
- If it's at rank 2, the reciprocal rank is 1/2 = **0.5**
- If it's at rank 3, the reciprocal rank is 1/3 = **0.33**
- If it never appears, the reciprocal rank is **0.0**
- MRR is just the average of these scores across all queries

**Why MRR is the most useful metric here:** Top-5 accuracy treats rank 1 and rank 5 as equally good. MRR rewards models that consistently put the right answer *first*. Two models can both achieve 100% top-5 accuracy, but the one with higher MRR is genuinely better at retrieval.

**Avg Top Score** — The average cosine similarity of the #1 result across all queries. Higher means the model is more "confident" in its top match.

In [ ]:
def retrieve_top_k(query: str, corpus_embeddings: list, model: str, k: int = 5):
    """Embed a query and retrieve the top-k most similar chunks.
    
    Returns a list of (chunk_index, similarity_score) tuples, sorted by
    similarity descending.
    """
    query_vec = get_embedding(query, model=model)
    
    similarities = []
    for i, chunk_vec in enumerate(corpus_embeddings):
        score = cosine_similarity(query_vec, chunk_vec)
        similarities.append((i, score))
    
    # Sort by similarity descending and return top k
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:k]


def reciprocal_rank(expected_sources: list[str], retrieved_sources: list[str]) -> float:
    """Compute the reciprocal rank of the first relevant result.
    
    Returns 1/rank where rank is the position (1-indexed) of the first
    chunk from an expected source. Returns 0.0 if no relevant result found.
    """
    for i, source in enumerate(retrieved_sources):
        if source in expected_sources:
            return 1.0 / (i + 1)
    return 0.0


def evaluate_model(model_name: str, corpus_embeddings: list, queries: list, k: int = 5):
    """Run all test queries against a model's embeddings and compute metrics.
    
    Returns a dict with:
      - top1_accuracy: fraction of queries where the #1 result was relevant
      - top5_accuracy: fraction of queries where an expected doc appeared in top-k
      - mrr: Mean Reciprocal Rank across all queries
      - avg_top_score: average similarity score of the #1 result
      - per_query: detailed per-query results
    """
    top1_hits = 0
    top5_hits = 0
    rr_scores = []
    top_scores = []
    per_query = []
    
    for tq in queries:
        results = retrieve_top_k(tq["query"], corpus_embeddings, model_name, k=k)
        
        # Get the source filenames for the top-k chunks
        top_sources = [all_chunks[idx].metadata["source"] for idx, _ in results]
        top_score = results[0][1] if results else 0.0
        top_scores.append(top_score)
        
        # Top-1: is the very first result from an expected source?
        top1_hit = top_sources[0] in tq["relevant_sources"] if top_sources else False
        if top1_hit:
            top1_hits += 1
        
        # Top-5: does any expected source appear in the top-k?
        top5_hit = any(src in top_sources for src in tq["relevant_sources"])
        if top5_hit:
            top5_hits += 1
        
        # Reciprocal rank
        rr = reciprocal_rank(tq["relevant_sources"], top_sources)
        rr_scores.append(rr)
        
        per_query.append({
            "query": tq["query"],
            "expected": tq["relevant_sources"],
            "retrieved": top_sources,
            "top_score": top_score,
            "top1_hit": top1_hit,
            "top5_hit": top5_hit,
            "reciprocal_rank": rr,
        })
    
    return {
        "top1_accuracy": top1_hits / len(queries) if queries else 0,
        "top5_accuracy": top5_hits / len(queries) if queries else 0,
        "mrr": np.mean(rr_scores) if rr_scores else 0,
        "avg_top_score": np.mean(top_scores) if top_scores else 0,
        "per_query": per_query,
    }

print("Retrieval functions defined.")

In [ ]:
# Run the evaluation for each model
model_results = {}

for model_name in MODEL_NAMES:
    print(f"Evaluating {model_name}...")
    result = evaluate_model(model_name, model_embeddings[model_name], test_queries, k=5)
    model_results[model_name] = result
    print(f"  Top-1 Accuracy: {result['top1_accuracy']:.0%}")
    print(f"  Top-5 Accuracy: {result['top5_accuracy']:.0%}")
    print(f"  MRR:            {result['mrr']:.4f}")
    print(f"  Avg Top Score:  {result['avg_top_score']:.4f}")
    print()

# Per-query detail with ranking info
for model_name in MODEL_NAMES:
    print(f"\n{'='*70}")
    print(f"  {model_name} — Per-Query Results")
    print(f"{'='*70}")
    for i, pq in enumerate(model_results[model_name]["per_query"]):
        difficulty = test_queries[i]["difficulty"]
        rank_label = f"rank 1" if pq["top1_hit"] else (f"rank {int(1/pq['reciprocal_rank'])}" if pq["reciprocal_rank"] > 0 else "MISS")
        marker = "  " if pq["top1_hit"] else ">>"
        print(f"  {marker}[{rank_label:>6}] ({difficulty:>6}) {pq['query']}")
        print(f"          Expected: {pq['expected']}")
        print(f"          Got:      {pq['retrieved'][:3]}")
        print(f"          Score: {pq['top_score']:.4f}  |  RR: {pq['reciprocal_rank']:.2f}")
        print()

# Metrics breakdown by difficulty tier
print(f"\n{'='*70}")
print(f"  Metrics by Difficulty Tier")
print(f"{'='*70}")

for difficulty in ["easy", "medium", "hard"]:
    tier_indices = [i for i, q in enumerate(test_queries) if q["difficulty"] == difficulty]
    print(f"\n  {difficulty.upper()} ({len(tier_indices)} queries):")
    print(f"  {'Model':<20} {'Top-1':>6} {'Top-5':>6} {'MRR':>8} {'Avg Score':>10}")
    print(f"  {'-'*52}")
    for model_name in MODEL_NAMES:
        pqs = [model_results[model_name]["per_query"][i] for i in tier_indices]
        t1 = sum(1 for pq in pqs if pq["top1_hit"]) / len(pqs)
        t5 = sum(1 for pq in pqs if pq["top5_hit"]) / len(pqs)
        mrr = np.mean([pq["reciprocal_rank"] for pq in pqs])
        avg_score = np.mean([pq["top_score"] for pq in pqs])
        print(f"  {model_name:<20} {t1:>5.0%} {t5:>5.0%} {mrr:>8.4f} {avg_score:>10.4f}")

---

## 6. Results Table

Side-by-side comparison of all three models across key metrics.

In [ ]:
# Print a formatted comparison table
header = f"{'Metric':<25}"
for m in MODEL_NAMES:
    header += f" {m:<18}"
print(header)
print("-" * (25 + 19 * len(MODEL_NAMES)))

# Dimensions
row = f"{'Dimensions':<25}"
for m in MODEL_NAMES:
    row += f" {MODELS[m]['dimensions']:<18}"
print(row)

# Top-1 Accuracy
row = f"{'Top-1 Accuracy':<25}"
for m in MODEL_NAMES:
    row += f" {model_results[m]['top1_accuracy']:<18.0%}"
print(row)

# Top-5 Accuracy
row = f"{'Top-5 Accuracy':<25}"
for m in MODEL_NAMES:
    row += f" {model_results[m]['top5_accuracy']:<18.0%}"
print(row)

# MRR
row = f"{'MRR':<25}"
for m in MODEL_NAMES:
    row += f" {model_results[m]['mrr']:<18.4f}"
print(row)

# Avg Top Score
row = f"{'Avg Top Score':<25}"
for m in MODEL_NAMES:
    row += f" {model_results[m]['avg_top_score']:<18.4f}"
print(row)

# Embedding Time
row = f"{'Embed Time (corpus)':<25}"
for m in MODEL_NAMES:
    row += f" {model_times[m]:<17.2f}s"
print(row)

# Cost Tier
row = f"{'Cost per 1M tokens':<25}"
for m in MODEL_NAMES:
    row += f" ${MODELS[m]['cost_per_million_tokens']:<17}"
print(row)

---

## 7. Visualization

Bar charts comparing the three models on retrieval accuracy, embedding time, and cost.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

colors = ["#4A90D9", "#50C878", "#E8832A"]
x = np.arange(len(MODEL_NAMES))
bar_width = 0.5

# --- Chart 1: Top-1 Retrieval Accuracy ---
top1_accs = [model_results[m]["top1_accuracy"] * 100 for m in MODEL_NAMES]
bars1 = axes[0].bar(x, top1_accs, bar_width, color=colors)
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Top-1 Accuracy")
axes[0].set_xticks(x)
axes[0].set_xticklabels(MODEL_NAMES, rotation=15, ha="right")
axes[0].set_ylim(0, 105)
for bar, val in zip(bars1, top1_accs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                 f"{val:.0f}%", ha="center", va="bottom", fontsize=10)

# --- Chart 2: MRR ---
mrrs = [model_results[m]["mrr"] for m in MODEL_NAMES]
bars2 = axes[1].bar(x, mrrs, bar_width, color=colors)
axes[1].set_ylabel("MRR")
axes[1].set_title("Mean Reciprocal Rank")
axes[1].set_xticks(x)
axes[1].set_xticklabels(MODEL_NAMES, rotation=15, ha="right")
axes[1].set_ylim(0, 1.1)
for bar, val in zip(bars2, mrrs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=10)

# --- Chart 3: Embedding Time ---
times = [model_times[m] for m in MODEL_NAMES]
bars3 = axes[2].bar(x, times, bar_width, color=colors)
axes[2].set_ylabel("Time (seconds)")
axes[2].set_title("Corpus Embedding Time")
axes[2].set_xticks(x)
axes[2].set_xticklabels(MODEL_NAMES, rotation=15, ha="right")
for bar, val in zip(bars3, times):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 f"{val:.2f}s", ha="center", va="bottom", fontsize=10)

# --- Chart 4: Cost per 1M Tokens ---
costs = [MODELS[m]["cost_per_million_tokens"] for m in MODEL_NAMES]
bars4 = axes[3].bar(x, costs, bar_width, color=colors)
axes[3].set_ylabel("USD per 1M tokens")
axes[3].set_title("Cost per 1M Tokens")
axes[3].set_xticks(x)
axes[3].set_xticklabels(MODEL_NAMES, rotation=15, ha="right")
for bar, val in zip(bars4, costs):
    axes[3].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                 f"${val}", ha="center", va="bottom", fontsize=10)

plt.suptitle("Voyage AI Embedding Model Comparison — Northbrook Corpus", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 8. Cost Analysis

Estimate the cost of embedding 1,000 documents for each model tier. We use the average character count from our corpus to approximate token counts.

**Approximation:** 1 token is roughly 4 characters of English text.

In [ ]:
# Calculate average document size from our corpus
avg_chars = np.mean([len(doc["text"]) for doc in documents])
avg_tokens_per_doc = avg_chars / 4  # rough approximation: 1 token ~ 4 chars

print(f"Corpus stats:")
print(f"  Documents loaded:       {len(documents)}")
print(f"  Avg chars per document: {avg_chars:,.0f}")
print(f"  Avg tokens per doc:     {avg_tokens_per_doc:,.0f} (estimated)")
print()

# Cost estimate for 1,000 documents
num_docs = 1_000
total_tokens = num_docs * avg_tokens_per_doc

print(f"Cost estimate for {num_docs:,} documents ({total_tokens:,.0f} estimated tokens):")
print(f"{'Model':<20} {'Cost/1M tokens':>15} {'Estimated Cost':>15}")
print("-" * 52)

for model_name in MODEL_NAMES:
    cost_per_m = MODELS[model_name]["cost_per_million_tokens"]
    estimated_cost = (total_tokens / 1_000_000) * cost_per_m
    print(f"{model_name:<20} ${cost_per_m:>14} ${estimated_cost:>14.4f}")

print()
print("Note: These are embedding costs only (one-time corpus ingestion).")
print("Query embedding costs are per-search and typically much smaller.")

In [ ]:
# Visualize cost comparison
fig, ax = plt.subplots(figsize=(8, 4))

estimated_costs = []
for model_name in MODEL_NAMES:
    cost_per_m = MODELS[model_name]["cost_per_million_tokens"]
    estimated_costs.append((total_tokens / 1_000_000) * cost_per_m)

bars = ax.bar(MODEL_NAMES, estimated_costs, color=colors, width=0.5)
ax.set_ylabel("Estimated Cost (USD)")
ax.set_title(f"Estimated Embedding Cost for {num_docs:,} Documents")

for bar, val in zip(bars, estimated_costs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f"${val:.4f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

---

## 9. Recommendation Template

Based on the data above, fill in your recommendation for which embedding model to use for the Northbrook Partners pipeline. There is no single right answer — defend your choice with data.

---

### My Recommendation

**Model choice:** [voyage-3-lite / voyage-3 / voyage-3-large]

**Justification:**

- **Top-1 Accuracy:** [Which model ranked the correct document first most often? Was the difference large enough to matter?]
- **MRR:** [Which model had the highest MRR? What does that tell you about ranking quality?]
- **Confidence (Avg Score):** [Did the larger model show higher similarity scores? Does confidence matter if ranking accuracy is similar?]
- **Cost:** [What is the cost difference between models? Is the quality improvement worth 3x or 9x the price?]
- **Use case fit:** [Northbrook has ~15 internal documents. At this scale, does the difference between models justify the cost? What if the corpus grew to 1,000 documents?]

**In 2-3 sentences:** [Write your final recommendation here. Cite specific numbers from the comparison. Explain the tradeoff you're making.]

---

## Summary

This notebook compared three Voyage AI embedding models on the same corpus and queries:

| What We Measured | Why It Matters |
|------------------|----------------|
| Top-1 accuracy | Does the model rank the right answer first? (Strictest test) |
| Top-5 accuracy | Does the right answer appear somewhere in the top 5? (Lenient baseline) |
| Mean Reciprocal Rank (MRR) | On average, how high is the first correct result? (Best overall quality signal) |
| Average similarity score | How confident is the model in its top match? |
| Embedding time | How long does corpus ingestion take? |
| Cost per 1M tokens | What does it cost at scale? |

### What We Learned

**"Bigger model = better" is not guaranteed.** On our corpus:
- All three models achieved 100% top-5 accuracy — if we only measured that, we'd conclude they're identical
- Top-1 accuracy and MRR revealed real differences in ranking quality
- The larger model had the highest confidence scores, but didn't always rank correctly
- The standard model sometimes outperformed the large model on certain query types

**The right metric changes the answer.** This is why evaluation methodology matters — a single accuracy number can hide important differences between models.

**The real skill is knowing how to test.** You now have a repeatable method: define queries with known answers, run retrieval, and compare models on multiple metrics. This is what you'd do in a real project before committing to a model.

This comparison feeds directly into Session 2.2, where you will commit to a model and use it to ingest chunks into ChromaDB.